
# TP — Classical RL Algorithms II: **Model-Free Methods** (Gridworld: CliffWalking)

**Objectifs pédagogiques**  
- Implémenter les méthodes **Monte Carlo** (prédiction & contrôle) ;
- Implémenter **TD(0)** avec **SARSA** (*on-policy*) ;
- Implémenter **Q-learning** (*off-policy*) ;
- Comparer les performances et l'impact de l’**exploration ε-greedy** et des **hyperparamètres** (`α`, `γ`, `ε`).

> **Environnement :** `CliffWalking-v1` (Gymnasium/Gym).  
> L’agent part en bas à gauche (S) et veut atteindre l’arrivée (G) en bas à droite. Tomber de la falaise donne **-100** et termine l’épisode ; chaque pas coûte **-1**.



## 🔧 Installation rapide (à exécuter localement si besoin)
```bash
# Optionnel selon votre environnement :
# pip install gymnasium matplotlib numpy
# ou, si vous utilisez Gym classique :
# pip install gym==0.26.2
```



## 📦 Imports & helpers (compatibilité Gymnasium/Gym)

On cherche à être compatibles à la fois avec **Gymnasium** (API récente) et **Gym** (API historique).


In [18]:

import math
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Callable, Tuple, List, Optional

# Compatibilité Gymnasium/Gym
try:
    import gymnasium as gym
    NEW_API = True
except Exception:
    import gym  # type: ignore
    NEW_API = False

ENV_ID = "CliffWalking-v1"

def make_env(env_id=ENV_ID, render=True):
    try:
        return gym.make(env_id, render_mode="human" if (render and NEW_API) else None)
    except TypeError:
        # Gym classique ne connaît pas render_mode
        return gym.make(env_id)

def env_reset(env, seed: Optional[int] = None):
    if NEW_API:
        try:
            return env.reset(seed=seed)
        except TypeError:
            return env.reset()
    else:
        if seed is not None:
            try:
                env.seed(seed)
            except Exception:
                pass
        s = env.reset()
        return (s, {})  # aligner la signature

def env_step(env, a):
    if NEW_API:
        s2, r, terminated, truncated, info = env.step(a)
        done = terminated or truncated
        return s2, r, done, info
    else:
        s2, r, done, info = env.step(a)
        return s2, r, done, info



## 🎯 Politique ε-greedy & plannings d'exploration / apprentissage

- **ε-greedy** : avec probabilité `ε`, on choisit une action aléatoire ; sinon l'action **greedy** (argmax de `Q[s]`).  
- **Plannings** (schedules) : `ε_t = ε0 / (1 + k t)` ; `α_t = α0 / (1 + k t)`.


In [19]:

def epsilon_greedy(Q: np.ndarray, state: int, nA: int, epsilon: float) -> int:
    if np.random.rand() < epsilon:
        return np.random.randint(nA)
    q_row = Q[state]
    max_q = np.max(q_row)
    best_actions = np.flatnonzero(np.isclose(q_row, max_q))
    return np.random.choice(best_actions)

def greedy_action(Q: np.ndarray, state: int) -> int:
    q_row = Q[state]
    max_q = np.max(q_row)
    best_actions = np.flatnonzero(np.isclose(q_row, max_q))
    return np.random.choice(best_actions)

def epsilon_schedule(epsilon0: float, k: float = 0.0) -> Callable[[int], float]:
    if k <= 0:
        return lambda t: epsilon0
    return lambda t: epsilon0 / (1.0 + k * t)

def alpha_schedule(alpha0: float, k: float = 0.0) -> Callable[[int], float]:
    if k <= 0:
        return lambda t: alpha0
    return lambda t: alpha0 / (1.0 + k * t)



## 🧮 Monte Carlo — **Prédiction** (every-visit)

On estime la **valeur d’état** \( V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t=s] \) pour une politique **fixe** \(\pi\).  
Principe : on génère des épisodes complets sous \(\pi\), on calcule les **retours** \(G_t\) et on moyenne (every-visit).

**Avantage :** pas besoin de modèle de l’environnement.  
**Limite :** nécessite des épisodes terminaux (on met à jour **après** chaque épisode).


In [20]:

def mc_prediction(env, policy: Callable[[int], int], gamma: float, episodes: int) -> np.ndarray:
    nS = env.observation_space.n
    V = np.zeros(nS, dtype=np.float64)
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for _ in range(episodes):
        episode = []
        s, _ = env_reset(env)
        done = False
        while not done:
            a = policy(s)
            s2, r, done, _ = env_step(env, a)
            episode.append((s, r))
            s = s2

        G = 0.0
        for t in reversed(range(len(episode))):
            s_t, r_t = episode[t]
            G = gamma * G + r_t
            returns_sum[s_t] += G
            returns_count[s_t] += 1

        for s_idx in range(nS):
            if returns_count[s_idx] > 0:
                V[s_idx] = returns_sum[s_idx] / returns_count[s_idx]

    return V



## 🕹️ Monte Carlo — **Contrôle on-policy** (ε-greedy, every-visit)

On apprend une **Q-fonction** et on améliore la politique **ε-greedy(Q)** au fil des épisodes.  
Mise à jour **incrémentale** every-visit :
\[
Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha \left(G_t - Q(s_t,a_t) \right).
\]


In [21]:

def mc_control_on_policy(env, episodes: int, gamma: float,
                         eps_sched: Callable[[int], float],
                         alpha_sched: Optional[Callable[[int], float]] = None,
                         track: bool = True):
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=np.float64)
    ep_rewards, ep_falls = [], []

    for ep in range(episodes):
        epsilon = eps_sched(ep)
        alpha = alpha_sched(ep) if alpha_sched else 1.0

        episode = []
        s, _ = env_reset(env)
        done = False
        fell = 0
        total = 0.0
        while not done:
            a = epsilon_greedy(Q, s, nA, epsilon)
            s2, r, done, _ = env_step(env, a)
            total += r
            if r <= -100:
                fell = 1
            episode.append((s, a, r))
            s = s2

        G = 0.0
        for t in reversed(range(len(episode))):
            s_t, a_t, r_t = episode[t]
            G = gamma * G + r_t
            Q[s_t, a_t] += alpha * (G - Q[s_t, a_t])

        if track:
            ep_rewards.append(total)
            ep_falls.append(fell)

    return Q, ep_rewards, ep_falls



## ⚡ TD(0) — **SARSA** (*on-policy*)

Mise à jour **au fil de l’eau** (bootstrapping) :
\[
Q(s,a) \leftarrow Q(s,a) + \alpha \Big(r + \gamma Q(s',a') - Q(s,a)\Big).
\]
La politique utilisée pour agir est la même que celle évaluée (**on-policy**, ici ε-greedy).


In [22]:

def sarsa_td0(env, episodes: int, gamma: float,
              eps_sched: Callable[[int], float],
              alpha_sched: Callable[[int], float],
              track: bool = True):
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=np.float64)
    ep_rewards, ep_falls = [], []

    for ep in range(episodes):
        epsilon = eps_sched(ep)
        alpha = alpha_sched(ep)
        s, _ = env_reset(env)
        a = epsilon_greedy(Q, s, nA, epsilon)
        done = False
        total, fell = 0.0, 0

        while not done:
            s2, r, done, _ = env_step(env, a)
            total += r
            if r <= -100:
                fell = 1
            a2 = epsilon_greedy(Q, s2, nA, epsilon) if not done else 0
            td_target = r + (0 if done else gamma * Q[s2, a2])
            Q[s, a] += alpha * (td_target - Q[s, a])
            s, a = s2, a2

        if track:
            ep_rewards.append(total)
            ep_falls.append(fell)

    return Q, ep_rewards, ep_falls



## 🧠 **Q-learning** (*off-policy*)

On met à jour vers la **meilleure** action suivante, indépendamment de l'action réellement choisie (off-policy) :
\[
Q(s,a) \leftarrow Q(s,a) + \alpha \Big(r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Big).
\]


In [23]:

def q_learning(env, episodes: int, gamma: float,
               eps_sched: Callable[[int], float],
               alpha_sched: Callable[[int], float],
               track: bool = True):
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=np.float64)
    ep_rewards, ep_falls = [], []

    for ep in range(episodes):
        epsilon = eps_sched(ep)
        alpha = alpha_sched(ep)
        s, _ = env_reset(env)
        done = False
        total, fell = 0.0, 0

        while not done:
            a = epsilon_greedy(Q, s, nA, epsilon)
            s2, r, done, _ = env_step(env, a)
            total += r
            if r <= -100:
                fell = 1
            max_next = 0.0 if done else np.max(Q[s2])
            td_target = r + gamma * max_next
            Q[s, a] += alpha * (td_target - Q[s, a])
            s = s2

        if track:
            ep_rewards.append(total)
            ep_falls.append(fell)

    return Q, ep_rewards, ep_falls



## 📏 Évaluation & tracés

- **Évaluation greedy** : on exécute la politique **greedy(Q)** (sans exploration) et on mesure la récompense moyenne + taux de chutes.  
- **Courbes** : (i) récompense par épisode, (ii) taux cumulé de chute.


In [24]:

def evaluate_greedy(env, Q: np.ndarray, episodes: int = 100) -> Tuple[float, float]:
    totals = []
    falls = 0
    for _ in range(episodes):
        s, _ = env_reset(env)
        done = False
        total = 0.0
        fell = False
        while not done:
            a = greedy_action(Q, s)
            s, r, done, _ = env_step(env, a)
            total += r
            if r <= -100:
                fell = True
        totals.append(total)
        falls += int(fell)
    return float(np.mean(totals)), falls / episodes

def smooth(x: List[float], k: int = 50) -> np.ndarray:
    if len(x) == 0:
        return np.array([])
    k = max(1, min(k, len(x)))
    w = np.ones(k) / k
    return np.convolve(np.array(x, dtype=float), w, mode="same")

def plot_learning_curve(rewards: List[float], title: str):
    plt.figure()
    plt.plot(rewards, label="Reward par épisode")
    s = smooth(rewards, k=max(5, len(rewards)//50))
    plt.plot(s, label="Lissé")
    plt.legend()
    plt.xlabel("Épisode")
    plt.ylabel("Récompense")
    plt.title(title)

def plot_fall_rate(falls: List[int], title: str):
    import numpy as np
    plt.figure()
    falls = np.array(falls, dtype=float)
    cum_rate = np.cumsum(falls) / (np.arange(len(falls)) + 1)
    plt.plot(cum_rate)
    plt.xlabel("Épisode")
    plt.ylabel("Taux cumulé de chute")
    plt.title(title)



## 🔬 Démo 1 — **MC Prédiction** d'une politique fixe (aléatoire)

On évalue une politique **uniforme aléatoire** (choix d'action équiprobable). On affiche la **fonction de valeur** \(V^\pi\).


In [ ]:

env = make_env(ENV_ID, render=True)

# Politique aléatoire uniforme (4 actions disponibles dans CliffWalking)
def random_policy(state: int) -> int:
    return np.random.randint(env.action_space.n)

V = mc_prediction(env, policy=random_policy, gamma=1.0, episodes=2000)
print("Valeur moyenne (quelques états) :", V[:10])

# Affichage en grille (CliffWalking: 4x12)
try:
    h, w = 4, 12
    plt.figure()
    plt.imshow(V.reshape(h, w))
    plt.colorbar()
    plt.title("MC Prédiction — V^π (politique aléatoire)")
except Exception as e:
    print("Affichage grille indisponible (reshape). Taille V:", V.shape, e)



## 🧪 Démo 2 — **Contrôle** : MC vs SARSA vs Q-learning

On entraîne chaque algorithme, puis on :
1. affiche la courbe de **récompense** par épisode ;
2. affiche le **taux de chutes** ;
3. évalue la **politique greedy(Q)** finale.


In [17]:

SEED = 42
np.random.seed(SEED)

episodes = 2000
gamma = 1.0
eps0 = 0.1
alpha0 = 0.5
eps_k = 0.0     # 0 = constant ; >0 = décroissant
alpha_k = 0.0

env = make_env(ENV_ID, render=False)
try:
    env_reset(env, seed=SEED)
except Exception:
    pass

eps_sched = epsilon_schedule(eps0, k=eps_k)
alpha_sched = alpha_schedule(alpha0, k=alpha_k)

results = {}
for algo in ["mc", "sarsa", "qlearning"]:
    if algo == "mc":
        Q, rewards, falls = mc_control_on_policy(env, episodes, gamma, eps_sched, alpha_sched, track=True)
    elif algo == "sarsa":
        Q, rewards, falls = sarsa_td0(env, episodes, gamma, eps_sched, alpha_sched, track=True)
    else:
        Q, rewards, falls = q_learning(env, episodes, gamma, eps_sched, alpha_sched, track=True)

    r_mean, fall_rate = evaluate_greedy(env, Q, episodes=200)
    results[algo] = {
        "Q": Q,
        "rewards": rewards,
        "falls": falls,
        "eval_reward_mean": r_mean,
        "eval_fall_rate": fall_rate,
    }
    plot_learning_curve(rewards, f"{algo.upper()} — Récompense par épisode")
    plot_fall_rate(falls, f"{algo.upper()} — Taux de chutes cumulé")

for algo, d in results.items():
    print(f"{algo.upper():10s} | Reward moyen (greedy x200): {d['eval_reward_mean']:.2f} | Taux de chutes: {100*d['eval_fall_rate']:.1f}%")


KeyboardInterrupt: 


## 📉 Étude 1 — Impact de **l'exploration** (ε constant vs décroissant)

On compare un ε **constant** (0.1) et un ε **décroissant** (ε₀=0.3 → décroissance `ε_t = ε0/(1+k t)`), pour **SARSA** et **Q-learning**.


In [ ]:

def run_and_log(env, algo: str, episodes: int, gamma: float, eps0: float, eps_k: float, alpha0: float, alpha_k: float, seed: int = 0):
    np.random.seed(seed)
    try:
        env_reset(env, seed=seed)
    except Exception:
        pass
    eps_sched = epsilon_schedule(eps0, k=eps_k)
    alpha_sched = alpha_schedule(alpha0, k=alpha_k)
    if algo == "sarsa":
        Q, rewards, falls = sarsa_td0(env, episodes, gamma, eps_sched, alpha_sched, track=True)
    else:
        Q, rewards, falls = q_learning(env, episodes, gamma, eps_sched, alpha_sched, track=True)
    r_mean, fall_rate = evaluate_greedy(env, Q, episodes=200)
    return rewards, falls, r_mean, fall_rate

env = make_env(ENV_ID, render=False)

EPISODES = 3000
configs = [
    ("sarsa", "ε constant 0.1", dict(eps0=0.1, eps_k=0.0)),
    ("sarsa", "ε décroissant (ε0=0.3, k=1e-3)", dict(eps0=0.3, eps_k=1e-3)),
    ("qlearning", "ε constant 0.1", dict(eps0=0.1, eps_k=0.0)),
    ("qlearning", "ε décroissant (ε0=0.3, k=1e-3)", dict(eps0=0.3, eps_k=1e-3)),
]

summary = []
for algo, label, params in configs:
    rewards, falls, r_mean, fall_rate = run_and_log(env, algo, EPISODES, gamma=1.0,
                                                    eps0=params["eps0"], eps_k=params["eps_k"],
                                                    alpha0=0.5, alpha_k=0.0, seed=SEED)
    plot_learning_curve(rewards, f"{algo.upper()} — {label}")
    plot_fall_rate(falls, f"{algo.upper()} — {label}")
    summary.append((algo, label, float(np.mean(rewards[-200:])), r_mean, float(fall_rate)))

import pandas as pd
df = pd.DataFrame(summary, columns=["Algo", "Exploration", "Reward moy. (derniers 200 ep)", "Reward moy. greedy (x200)", "Taux de chutes greedy"])
df



## 🔎 Étude 2 — Mini *grid-search* sur `α` et `ε` (Q-learning)

On balaye quelques couples `(α, ε)` et on observe la performance **greedy** finale.


In [ ]:

env = make_env(ENV_ID, render=False)
alphas = [0.1, 0.3, 0.5]
epsilons = [0.05, 0.1, 0.3]

rows = []
for a in alphas:
    for e in epsilons:
        rewards, falls, r_mean, fall_rate = run_and_log(env, "qlearning", episodes=2000, gamma=1.0,
                                                        eps0=e, eps_k=0.0, alpha0=a, alpha_k=0.0, seed=SEED)
        rows.append((a, e, float(np.mean(rewards[-200:])), r_mean, float(fall_rate)))

import pandas as pd
df_grid = pd.DataFrame(rows, columns=["alpha", "epsilon", "Reward moy. (derniers 200 ep)", "Reward moy. greedy (x200)", "Taux de chutes greedy"])
df_grid.sort_values(by="Reward moy. greedy (x200)", ascending=False)



## 🧭 Conseils & points d’attention

- **CliffWalking** est **très punitif** sur la falaise (-100).  
  - **SARSA (on-policy)** tend à apprendre des chemins **plus sûrs** (s'éloigne du bord).  
  - **Q-learning (off-policy)** peut favoriser des routes plus **agressives** près de la falaise pour maximiser la valeur estimée.
- **MC** doit attendre la **fin de l'épisode** (retours) ; **TD(0)** met à jour **à chaque pas** (bootstrapping).
- Pendant l’**évaluation**, utilisez une **politique greedy** (ε=0) pour mesurer la performance réelle de la politique apprise.
- **Hyperparamètres** : commencez avec `γ=1.0`, `α=0.5`, `ε=0.1`, puis ajustez en observant les courbes.
- **Décroissance d’ε** : utile pour explorer au début puis **exploiter** ensuite.
